# EN3150 Assignment 03 — Model B (Lightweight depthwise-separable CNN)

| Requirement from the assignment | Where it is handled |
|---|---|
| Replace standard convolutions with **depthwise separable** convolutions | Section 2 (`DSConv` in `a03_common.py`) |
| **Total trainable parameters <= 100,000** | Section 3 (hand calculation + `assert`) |
| Detail kernel sizes, filters, FC units; calculate total parameters | Section 2 table |
| Hardware-aware activation justification | Section 2 |
| Train >= 20 epochs, loss curves, test accuracy, confusion matrix, precision, recall | Sections 4-5 |
| Parameters, size (KB), time per epoch, accuracy for the comparison table | Section 6 |

**Before running:** keep `a03_common.py` in the same folder as this notebook, and run the **Model A notebook first** (it creates `outputs/eurosat_64.npz` and `outputs/splits.npz`, so every model uses the identical 70/15/15 split).

## 1. Setup
Same data, split, normalisation, optimizer (Adam, lr = 1e-3) and epoch count as Model A, so the comparison isolates the architecture.

In [ ]:

from a03_common import *      # also brings np, torch, nn, plt, os, json, time into scope

EPOCHS = 25
LR = 1e-3

set_seed(SEED)
print("Device:", DEVICE)
data = Data(norm="dataset", batch_size=64, augment=True)
print("Train/Val/Test:", len(data.idx["train"]), len(data.idx["val"]), len(data.idx["test"]))
print("Classes:", data.classes)

: 

## 2. Architecture

**Depthwise separable convolution** factorises one standard convolution into two cheaper steps:
1. **Depthwise 3x3** - one 3x3 filter per input channel (`groups = C_in`): filters space only.
2. **Pointwise 1x1** - mixes the channels (`C_in -> C_out`).

For an H x W output, kernel K x K:

| | Parameters | MACs |
|---|---|---|
| Standard conv | K²·C_in·C_out | H·W·K²·C_in·C_out |
| Depthwise separable | K²·C_in + C_in·C_out | H·W·(K²·C_in + C_in·C_out) |

Cost ratio = **1/C_out + 1/K²**, about 1/9 for K = 3 and wide layers (up to roughly 8-9x cheaper).

**Network** (spatial schedule identical to Model A: 64 -> 32 -> 16 -> 8 -> 4):

| Layer | Config | Output | DW weights | PW weights | BN (2 per channel) | Total |
|---|---|---|---|---|---|---|
| DSConv1 + MaxPool | 3 -> 32 | 32x32x32 | 27 | 96 | 70 | **193** |
| DSConv2 + MaxPool | 32 -> 64 | 64x16x16 | 288 | 2,048 | 192 | **2,528** |
| DSConv3 | 64 -> 128 | 128x16x16 | 576 | 8,192 | 384 | **9,152** |
| DSConv4 + MaxPool | 128 -> 128 | 128x8x8 | 1,152 | 16,384 | 512 | **18,048** |
| DSConv5 + MaxPool | 128 -> 256 | 256x4x4 | 1,152 | 32,768 | 768 | **34,688** |
| Global average pool | - | 256 | 0 | 0 | 0 | 0 |
| FC (logits) | 256 -> 10 | 10 | - | - | - | (256 + 1)·10 = **2,570** |
| **Total** | | | | | | **67,179** |

All depthwise kernels are 3x3, all pointwise kernels 1x1; conv layers have no bias because BatchNorm follows (BN supplies the shift).

**Design decisions and hardware-aware justification**
- **ReLU6** = `min(max(0, x), 6)`: only comparisons (no multiplications, no exponentials), and the bounded output range suits int8/fixed-point quantisation. Sigmoid, tanh, Swish and GELU need `exp()`, which is expensive on MCUs and Raspberry Pi class CPUs.
- **BatchNorm** after every conv: depthwise-separable nets train poorly without it. At inference BN folds into the preceding conv weights, so it costs nothing at run time. Its 2 trainable parameters per channel **are counted** in the 67,179.
- **Global average pooling + one FC** replaces Flatten + FC(4096 -> 128). That single layer held 524k parameters in Model A; here the whole classifier is 2,570.
- **Max-pooling** is kept for downsampling (parameter-free, comparison-only).
- Every convolution, including the first, is depthwise separable (strict reading of the brief). A common alternative keeps a standard stem conv.

**Fairness note for the report:** Model B differs from Model A in more than the convolution type (it also uses BN, ReLU6 and a GAP head). State this when you discuss the trade-offs.

In [ ]:
set_seed(SEED)
modelB = ModelB(data.num_classes, 3).to(DEVICE)
print(modelB)
print()
total_params, total_macs = summarize(modelB)

n_total, n_trainable = count_params(modelB)
print(f"\nTrainable parameters: {n_trainable:,}  (limit 100,000)")
assert n_trainable <= 100_000, "Model B exceeds the 100k parameter limit!"
if data.num_classes == 10:
    assert n_trainable == 67_179, "Does not match the hand calculation in the table"
print("Constraint satisfied (<= 100,000) and matches the hand calculation.")

**How much did the depthwise-separable factorisation save?** The cell below recomputes what *the same layer widths* would cost as standard 3x3 convolutions, which is a clean number for the report.

In [ ]:
size = IMG_SIZE
std_w = ds_w = std_macs = ds_macs = 0
for m in modelB.features:
    if isinstance(m, DSConv):
        cin, cout = m[0].in_channels, m[3].out_channels
        std_w += 9 * cin * cout;            ds_w += 9 * cin + cin * cout
        std_macs += size * size * 9 * cin * cout
        ds_macs += size * size * (9 * cin + cin * cout)
    elif isinstance(m, nn.MaxPool2d):
        size //= 2
print(f"Conv weights  : standard {std_w:>10,} | depthwise separable {ds_w:>8,} | {std_w/ds_w:.1f}x fewer")
print(f"Conv MACs     : standard {std_macs:>10,} | depthwise separable {ds_macs:>8,} | {std_macs/ds_macs:.1f}x fewer")

## 3. Training
Adam (lr = 1e-3), cross-entropy, 25 epochs, best-validation checkpoint (the test set is never used for selection).

In [ ]:
set_seed(SEED)
modelB = ModelB(data.num_classes, 3).to(DEVICE)
optB = torch.optim.Adam(modelB.parameters(), lr=LR)
histB = train_model(modelB, optB, data, epochs=EPOCHS, name="Model B / Adam")

## 4. Learning curves
Compare with Model A: does B converge more slowly or more smoothly? Is the train-validation gap smaller (B has fewer parameters and BN regularisation, so overfitting is usually milder)?

In [ ]:
plot_history(histB, "Model B (Adam, lr=1e-3)", "modelB_curves.png")

## 5. Test-set evaluation
Accuracy, macro precision/recall, per-class report and confusion matrix. Use the test set once, after training.

In [ ]:
evB = evaluate(modelB, data)
cmB = plot_confusion(evB["y_true"], evB["y_pred"], data.classes,
                     f"Model B - confusion matrix (test acc {evB['acc']:.2%})", "modelB_confusion.png")

## 6. Deployment metrics and comparison with Model A
Size is the float32 `state_dict` file on disk. Latency is single-image CPU time on this laptop (relative comparison only). Everything is saved to `outputs/modelB_results.json`.

In [ ]:
resB = save_results("modelB", "Model B (depthwise separable)", modelB, histB, evB,
                    "Adam lr=1e-3", data)

In [ ]:
pa = OUT_DIR / "modelA_results.json"
if pa.exists():
    A = json.load(open(pa))
    rows = [("Parameters", A["params"], resB["params"], "{:,.0f}"),
            ("MACs", A["macs"], resB["macs"], "{:,.0f}"),
            ("Size on disk (KB)", A["size_kb"], resB["size_kb"], "{:,.1f}"),
            ("Time / epoch (s)", A["time_per_epoch_s"], resB["time_per_epoch_s"], "{:,.1f}"),
            ("CPU latency (ms/img)", A["latency_ms_cpu"], resB["latency_ms_cpu"], "{:,.2f}"),
            ("Test accuracy", A["test_acc"], resB["test_acc"], "{:.4f}")]
    print(f"{'Metric':<24}{'Model A':>14}{'Model B':>14}{'A / B':>10}")
    for name, a, bb, fmt in rows:
        print(f"{name:<24}{fmt.format(a):>14}{fmt.format(bb):>14}{a/bb:>10.2f}")
else:
    print("Run the Model A notebook first to see the side-by-side comparison.")

## 7. What to write in the report (use your printed numbers)
- **Part 2:** the layer table from Section 2, the **67,179** total, the 100k-limit check, and the ReLU6 / BN-folding / GAP justification.
- **Part 4 trade-offs (A vs B):**
  1. *Parameters, size, MACs* - quote the ratios from the A/B table. The parameter and MAC ratios are the "drastic reduction" the brief asks about.
  2. *Accuracy* - the accuracy gap between A and B (positive or negative) and which classes lose the most (confusion matrix).
  3. *Time per epoch* - **do not assume it falls as much as the MACs.** Depthwise convolutions have low arithmetic intensity (little compute per byte moved) and PyTorch's CPU kernels are less optimised for them, and BN/ReLU6 add memory traffic, so wall-clock gains are usually far smaller than the MAC ratio. This is a genuine finding: mention it, and note that deployment runtimes with optimised depthwise kernels (e.g. TFLite/XNNPACK) realise the MAC savings much better.
  4. *Convergence* - compare curves from Section 4.
- Keep the fairness note (BN + ReLU6 + GAP also differ from Model A).